# Revised Experiment 2 pilot and GPU qualification

Set `SOURCE_SHA` to the reviewed implementation commit before pushing. Default mode qualifies four full-size/full-10k-raw-step branches, full plasticity, learning-branch restart and measured solo memory before admitting a concurrent stress on each detected GPU. Reference qualification training is 10k raw steps at one checkpoint, so this is engineering evidence only. Pilot mode schedules the complete twenty-run/million-step/fifteen-checkpoint design. Outputs/logs stay in `/kaggle/working/exp2-revised`; preserve them between Kaggle sessions. No GPU model is assumed. Concurrency never exceeds two independent processes per GPU and falls back to one without matching measured evidence. Runtime dependencies follow the previously working Kaggle setup, including its Python 3.13 compatibility substitutions; this differs from the pinned local CPU environment.


In [ ]:
import os
os.environ.update({"JAX_PLATFORMS": "cuda,cpu", "MUJOCO_GL": "egl", "PYOPENGL_PLATFORM": "egl", "EGL_PLATFORM": "surfaceless", "MUJOCO_EGL_DEVICE_ID": "0", "WANDB_MODE": "disabled", "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1", "XLA_PYTHON_CLIENT_PREALLOCATE": "false"})
import signal
def _qualification_timeout(signum, frame):
    raise TimeoutError("Experiment 2 pilot GPU qualification exceeded its three-hour limit")
signal.signal(signal.SIGALRM, _qualification_timeout)
signal.alarm(11 * 60 * 60)
import platform
import subprocess
import sys
from pathlib import Path

SOURCE_SHA = os.environ.get("EXP2_SOURCE_SHA", "a7b229335b9737ab1ea1fe1b9f428bbc3ea26a6a")
SOURCE_BRANCH = "codex/exp2-online-prototype-20261107"
REPO = Path("/kaggle/temp/scaling-drl-research")
REPO.parent.mkdir(parents=True, exist_ok=True)
# Kaggle's internet toggle is required for both source and dependency downloads.
import socket
import time
import tempfile
import shutil

def wait_for_dns(host, attempts=6, delay=10):
    for attempt in range(attempts):
        try:
            socket.getaddrinfo(host, 443, type=socket.SOCK_STREAM)
            print("DNS available:", host)
            return
        except socket.gaierror as exc:
            print(f"DNS attempt {attempt + 1}/{attempts} failed for {host}: {exc}", flush=True)
            if attempt + 1 < attempts:
                time.sleep(delay)
    raise RuntimeError(f"Kaggle cannot resolve {host}. Check that Internet is enabled in notebook settings; if it is, retry after Kaggle networking recovers.")

for host in ("github.com", "pypi.org", "files.pythonhosted.org"):
    wait_for_dns(host)

if not REPO.exists():
    for attempt in range(3):
        with tempfile.TemporaryDirectory(prefix="exp2-clone-", dir=REPO.parent) as tmp:
            destination = Path(tmp) / "source"
            result = subprocess.run(["git", "clone", "--quiet", "--single-branch", "--branch", SOURCE_BRANCH, "https://github.com/kaves29/scaling-drl-research.git", str(destination)], text=True, capture_output=True)
            if result.returncode == 0:
                shutil.move(str(destination), str(REPO))
                break
            print(f"Git clone attempt {attempt + 1}/3 failed: {result.stderr.strip()}", flush=True)
        if attempt < 2:
            time.sleep(10)
    else:
        raise RuntimeError("Could not clone the pinned source after three attempts; check Kaggle internet access and GitHub availability.")
subprocess.run(["git", "checkout", "--quiet", "--detach", SOURCE_SHA], cwd=REPO, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == SOURCE_SHA
requirements = (REPO / "main/requirements.txt").read_text()
install_dm_control_without_optional_mazes = sys.version_info >= (3, 13)
if install_dm_control_without_optional_mazes:
    replacements = {"numpy==1.26.4": "numpy==2.1.3", "pandas==2.1.4": "pandas==2.2.3", "matplotlib==3.9.0": "matplotlib==3.9.2", "Shimmy==2.0.0": "Shimmy==1.3.0"}
    for old, new in replacements.items():
        assert requirements.count(old) == 1, f"Expected exactly one {old} pin"
        requirements = requirements.replace(old, new)
    for unused_pin in ("labmaze==1.0.6", "dm_control==1.0.38"):
        assert requirements.count(unused_pin) == 1, f"Expected exactly one {unused_pin} pin"
        requirements = requirements.replace(unused_pin + "\n", "")
runtime_requirements = Path("/kaggle/temp/exp2-runtime-requirements.txt")
runtime_requirements.write_text(requirements)
print("Python runtime:", platform.python_version(), "; installing runtime requirements:", runtime_requirements)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(runtime_requirements)], check=True)
if install_dm_control_without_optional_mazes:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "dm_control==1.0.38"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "jax-cuda13-plugin", "jax-cuda13-pjrt"], check=True, stdout=subprocess.DEVNULL)
    print("Removed Kaggle CUDA 13 JAX plugins to prevent conflicts with the pinned CUDA 12 backend.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "jax[cuda12]==0.4.34"], check=True)

compat = Path("/kaggle/temp/exp2-python-compat")
compat.mkdir(exist_ok=True)
(compat / "sitecustomize.py").write_text("import numpy as np\nif not hasattr(np, 'issctype'):\n    np.issctype = lambda rep: isinstance(rep, type) and issubclass(rep, np.generic)\n")
os.environ["PYTHONPATH"] = str(compat) + os.pathsep + str(REPO / "main")
os.environ["WANDB_MODE"] = "online"
# Never print credentials. Kaggle Secrets must expose the established W&B account.
from kaggle_secrets import UserSecretsClient
if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
assert os.environ.get("WANDB_API_KEY"), "Configure WANDB_API_KEY in Kaggle Secrets"
import wandb
wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)
api = wandb.Api()
os.environ["WANDB_ENTITY"] = api.default_entity
assert os.environ["WANDB_ENTITY"], "No established W&B entity accessible"
print("W&B established entity verified; project: Experiment-2-Pilot-Testing")


In [ ]:
import json
import subprocess
import time
from pathlib import Path

# qualification tests both regimes concurrently on every actual GPU, two jobs/GPU.
# pilot consumes that exact-source receipt and schedules the twenty approved runs.
MODE = "qualification"
OUTPUT = Path("/kaggle/working/exp2-revised")
OUTPUT.mkdir(exist_ok=True)
run_env = dict(os.environ, XLA_PYTHON_CLIENT_PREALLOCATE="false")

def inventory():
    text = subprocess.check_output(["nvidia-smi", "--query-gpu=index,uuid,name,memory.total,memory.free,memory.used", "--format=csv,noheader,nounits"], text=True)
    return [dict(zip(("index","uuid","name","total","free","used"), [x.strip() for x in line.split(",")])) for line in text.splitlines()]

initial = inventory()
assert initial, "No NVIDIA GPU detected"
print("Detected devices:", [{k:g[k] for k in ("index","name","total","free")} for g in initial])
receipt_path = OUTPUT / "concurrency_qualification.json"
if MODE == "qualification":
    # Memory admission must measure fresh work, not reload an already complete run.
    ATTEMPT = OUTPUT / ("qualification_attempt_" + str(time.time_ns()))
    ATTEMPT.mkdir()
    def launch(gpu, regime, phase):
        directory = ATTEMPT / (phase + "_gpu" + gpu["index"] + "_" + regime)
        directory.mkdir(exist_ok=True)
        log = (directory / "process.log").open("a")
        env = dict(run_env, CUDA_VISIBLE_DEVICES=gpu["index"], MUJOCO_EGL_DEVICE_ID="0")
        process = subprocess.Popen([sys.executable,"-m","experiments.exp12.revised_exp2_qualification","--regime",regime,"--output",str(directory)], env=env, stdout=log, stderr=subprocess.STDOUT)
        return (process, log, directory, gpu)

    def monitor(active):
        peaks = {gpu["uuid"]:0 for gpu in initial}
        last_report = 0
        while any(process.poll() is None for process,_,_,_ in active):
            for gpu in inventory():
                peaks[gpu["uuid"]] = max(peaks[gpu["uuid"]], float(gpu["used"]))
            if time.monotonic() - last_report > 60:
                print("Qualification running; peak MiB:", peaks, flush=True)
                last_report = time.monotonic()
            time.sleep(5)
        failures = []
        for process,log,directory,gpu in active:
            log.close()
            if process.returncode:
                failures.append({"directory":str(directory),"exit":process.returncode})
        assert not failures, f"Qualification failed; inspect process.log: {failures}"
        return peaks

    # First measure one full-size large-reference run per actual GPU. It includes
    # both shadow architectures, learning/frozen branches, assay, and restart.
    # Only enable the two-process stress after its measured memory fits twice.
    solo = [launch(gpu,"reference_large","qualification_solo") for gpu in initial]
    solo_peaks = monitor(solo)
    current = {gpu["uuid"]:gpu for gpu in inventory()}
    admitted = {gpu["uuid"]:(2 if 2 * solo_peaks[gpu["uuid"]] <= float(current[gpu["uuid"]]["free"]) else 1) for gpu in initial}
    print("Measured concurrency admission:", admitted)
    active = []
    for gpu in initial:
        if admitted[gpu["uuid"]] == 2:
            for regime in ("reference_small","reference_large"):
                active.append(launch(gpu,regime,"qualification_concurrent"))
        else:
            # Large-reference solo already passed. Complete the other regime
            # sequentially and retain the one-job ceiling for this device.
            active.append(launch(gpu,"reference_small","qualification_sequential"))
    final_peaks = monitor(active)
    receipts = [json.loads((directory / "qualification_result.json").read_text()) for _,_,directory,_ in solo + active]
    assert all(r["source_commit"] == SOURCE_SHA and r["full_branch_horizon"] and r["plasticity_probe"] and r["restart"] for r in receipts)
    receipt = {"source_commit":SOURCE_SHA,"full_branch_horizon":True,"plasticity_probe":True,"restart":True,"concurrent_pass":any(count == 2 for count in admitted.values()),
        "devices":{gpu["uuid"]:{"jobs":admitted[gpu["uuid"]],"solo_peak_mib":solo_peaks[gpu["uuid"]],"concurrent_peak_mib":final_peaks[gpu["uuid"]],"qualified_workloads": [[{"environment":"dog-run","reference_regime":"reference_small"},{"environment":"dog-run","reference_regime":"reference_large"}]] if admitted[gpu["uuid"]] == 2 else []} for gpu in initial},
        "runs":receipts,"reference_duration_limitation":"10k reference raw steps, one checkpoint; not million-step scientific qualification"}
    receipt_path.write_text(json.dumps(receipt,indent=2))
    print("Full branch/probe/restart qualification passed; per-device admitted concurrency:", admitted, "Receipt:", receipt_path)
elif MODE == "pilot":
    # Restore the exact-source qualification receipt/checkpoints from prior Kaggle output
    # into OUTPUT when continuing another session. Scheduler defaults to one job/GPU
    # when receipt is absent or devices/source/memory differ.
    command = [sys.executable,"-m","experiments.exp12.revised_exp2_scheduler","--output",str(OUTPUT / "pilot"),"--jobs-per-gpu","2"]
    if receipt_path.exists():
        command += ["--qualification",str(receipt_path)]
    with (OUTPUT / "scheduler.log").open("a") as log:
        subprocess.run(command,env=run_env,stdout=log,stderr=subprocess.STDOUT,check=True)
    print("All twenty runs completed; durable outputs:", OUTPUT / "pilot")
else:
    raise ValueError("MODE must be qualification or pilot")
